In [103]:
import numpy as np
import pandas as pd
import re

In [104]:
clues = pd.read_csv("xd/clues.tsv", sep = "\t", skiprows=[2627538])

In [105]:
clues = clues.dropna()

In [106]:
clues.head(10)

,pubid,year,answer,clue
0,7xw,2021,1138,THX ___ (George Lucas film referenced in many ...
1,7xw,2021,125243,Prime whose first three digits are p^q and who...
2,7xw,2021,158719,Multiple of XI (except not in Roman numerals; ...
3,7xw,2021,1812,Overture whose use of cannons is canon
4,7xw,2021,1IRON,"Not even God can hit the ___ --Lee Trevino, af..."
5,7xw,2021,1UP,Virtual life
6,7xw,2021,2255887,"Phone number that, on ""Scrubs"", could (aptly) ..."
7,7xw,2021,328509,69 cubed (nice)
8,7xw,2021,390000,Number that equals p^8 - p^4 for some prime p ...
9,7xw,2021,403403,"When halved (?!?), smallest semiprime whose pr..."


In [107]:
clues.sample(10)

,pubid,year,answer,clue
6414353,usa,1995,EGO,Trip of a sort
3353214,nyt,1960,STRAY,Lost lamb.
3655771,nyt,1970,ONE,Undivided
1723949,lat,2017,ALA,Mobile's st.
6566178,usa,2003,CXI,"111, in old Rome"
5479374,pk,2007,ORT,Meal morsel
1838495,lat,2020,TICS,Tourette syndrome symptoms
7070296,usa,2025,ATE,Had sushi
3265060,nyt,1957,VOGUES,Fashions.
1381175,lat,2005,PASSES,Lothario's array


In [108]:
clues.pubid.value_counts()

pubid
nyt       2443925
nw         840746
lat        832437
up         714350
usa        683388
cs         504736
wsj        392375
atc        168903
nys        147295
wap        118878
bg         110137
beq        109791
pk         101217
tny         60952
che         58650
club        55995
avc         42659
onion       24331
awm         22734
slate       22702
ss          18219
rnr         17912
gfp         16578
sl          13933
cca         13804
ross        12453
atl         10117
cc           9641
eltana       8833
7xw          4596
cht          3788
db           3722
mm           3562
tm           3535
upc          3447
mg           1452
tpfat        1179
nym           696
tmdcr         538
pzz            78
nyh            74
kc             43
nyw            31
Name: count, dtype: int64

In [109]:
clues_selected = clues[(clues.year >= 1995) & clues.pubid.isin(["nyt","usa","wsj","up"])]

In [110]:
x_row = re.compile("X+")

In [111]:
letter_row = re.compile("[A-Z]+")

In [112]:
x_clues = [bool(letter_row.fullmatch(str(s))) and not bool(x_row.fullmatch(str(s))) for s in clues_selected.answer]

In [113]:
clues_selected = clues_selected[x_clues]

In [114]:
clues_reference = re.compile(r"\d+( |-)([aA]cross|[dD]own)")

In [115]:
clues_selected = clues_selected[[not bool(clues_reference.findall(x)) for x in clues_selected.clue]]

In [116]:
clues_selected.sample(10)

,pubid,year,answer,clue
6156269,up,2015,SHOT,"Word with ""put,"" ""glass"" or ""clock"""
5043982,nyt,2014,AGNUS,___ Dei
4498957,nyt,1997,SALE,Red-tag event
5097125,nyt,2015,MIA,Farrow or Hamm
4630212,nyt,2001,REINS,Controlling interests?
5293452,nyt,2021,YEARNINGS,Wishes
4772527,nyt,2006,AMES,"Rachel of ""General Hospital"""
6748502,usa,2012,IDOS,A wedding requires two of them
5060465,nyt,2014,KEEN,What was once cool?
5249840,nyt,2020,NET,Court divider


In [117]:
clues_selected = clues_selected.copy()

In [118]:
clues_selected["length"] = [len(str(x)) for x in clues_selected.answer]

In [119]:
clues_selected.answer.unique()

array(['AAA', 'AAH', 'AANDP', ..., 'XMASLIST', 'YELLOWBELLPEPPER',
       'YELLOWFINISH'], shape=(188219,), dtype=object)

In [120]:
clues_selected.sample(10)

,pubid,year,answer,clue,length
4777074,nyt,2006,CORONA,Eclipse phenomenon,6
4439202,nyt,1995,STEELYARD,_____ Blues” (1973 Fonda film)\nnyt\t1995\tSTE...,9
5751522,up,2001,IMBUE,Permeate,5
6827074,usa,2015,IWO,___ Jima,3
7256253,wsj,2004,ELSA,Lohengrin's love,4
7319570,wsj,2013,AKIN,Similar,4
5902620,up,2006,SHIEST,Most withdrawn,6
6277896,up,2020,RED,Common color for the flower that this puzzle h...,3
6089110,up,2013,HOPED,Had aspirations,5
4944344,nyt,2011,IKEA,Low-priced furniture source,4


In [121]:
clues_selected["clue_count"] = [int(x) for x in clues_selected.groupby("answer").transform("count").clue]

In [122]:
clues_selected.sample(10)

,pubid,year,answer,clue,length,clue_count
4678311,nyt,2003,BRAZOS,Texas river,6,3
7480777,wsj,2021,ASTORS,Wealthy turn-of-the-century family,6,22
6512884,usa,2000,LESSTIME,In ___: quicker than,8,1
6302368,up,2021,SCENE,Public tantrum,5,386
6518287,usa,2000,SCOFF,Mock,5,50
5110717,nyt,2015,YEE,___-haw!,3,47
4612513,nyt,2001,DAD,Pops,3,453
7438300,wsj,2019,LOU,Baseball's Brock,3,470
5239001,nyt,2020,DEMETRI,Comedian Martin,7,1
4557702,nyt,1999,NISAN,Month after Adar,5,55


In [123]:
clues_selected = clues_selected[clues_selected.clue_count >= 10]

In [124]:
clues_selected.shape

(2407547, 6)

In [125]:
import tqdm

In [126]:
clues_ten = pd.concat([x.sample(10) for _,x in tqdm.tqdm(clues_selected.groupby("answer"))])

100%|███████████████████████████████████| 32475/32475 [00:12<00:00, 2703.18it/s]


In [127]:
clues_ten.sample(10)

,pubid,year,answer,clue,length,clue_count
5745468,up,2001,DANIEL,He was a Boone to pioneers?,6,41
4579503,nyt,2000,DEBASE,Pollute,6,46
7241392,wsj,2002,DEPP,Blow actor,4,100
7515953,wsj,2022,ISURECAN,Consider it done!,8,12
6619810,usa,2005,TROLL,The Three Billy Goats Gruff meanie,5,134
7270286,wsj,2006,ENCS,"SASEs, e.g.",4,15
5919841,up,2007,HOVE,Britain's Brighton and ___,4,21
4911094,nyt,2010,GETSTO,Bugs,6,18
5873327,up,2005,RUBON,"Apply, as suntan lotion",5,12
7246781,wsj,2002,UPSY,Daisy lead-in,4,16


In [128]:
clues_ranks = clues_ten[["answer","clue_count","length"]].groupby("answer").min()
clues_ranks["random_count"]=clues_ranks.clue_count + np.random.random([len(clues_ranks.clue_count)])
clues_ranks["common_rank"] = clues_ranks.groupby("length")["random_count"].rank(method="first").astype(int)

In [129]:
clues_ranks[clues_ranks.length == 5].common_rank.value_counts().value_counts()

count
1    8734
Name: count, dtype: int64

In [130]:
clues_ranks[clues_ranks.length == 5].common_rank.max()

8734

In [131]:
clues_ranks["length_max"]=clues_ranks.groupby("length")["common_rank"].transform("max")

In [132]:
clues_ranks["clue_pct"] = clues_ranks.common_rank/clues_ranks.length_max

In [133]:
px.histogram(clues_ranks.clue_pct)

In [134]:
clues_ranks[clues_ranks.common_rank == clues_ranks.length_max].sort_values("length")

,clue_count,length,random_count,common_rank,length_max,clue_pct
answer,,,,,,
ERA,2853,3,2853.784290,2264,2264,1.0
AREA,2689,4,2689.444088,5540,5540,1.0
ARENA,1344,5,1344.131448,8734,8734,1.0
ESTATE,324,6,324.882894,8358,8358,1.0
ESTATES,104,7,104.534541,4595,4595,1.0
ONEONONE,63,8,63.751021,1785,1785,1.0
HOTPOTATO,32,9,32.737358,414,414,1.0
ROUNDROBIN,29,10,29.695046,398,398,1.0
STOOLPIGEON,35,11,35.755662,146,146,1.0


In [135]:
import plotly.express as px

In [136]:
px.histogram(clues_ranks.clue_count)

In [137]:
clues_ten["clue_id"] = clues_ten.groupby("answer")["clue"].rank("first").astype(int)

In [138]:
clues_clean = pd.pivot(clues_ten, columns="clue_id", index="answer", values = "clue")

In [139]:
clues_clean.columns = [f"clue_{n+1}" for n in range(10)]

In [91]:
clues_clean = clues_clean.reset_index()

In [92]:
clues_clean = clues_clean.reset_index()

In [141]:
clues_ranks

,clue_count,length,random_count,common_rank,length_max,clue_pct
answer,,,,,,
AAA,553,3,553.058482,2024,2264,0.893993
AAAA,19,4,19.131588,703,5540,0.126895
AAAS,67,4,67.373217,2335,5540,0.421480
AABA,16,4,16.281466,523,5540,0.094404
AAH,415,3,415.490774,1857,2264,0.820230
...,...,...,...,...,...,...
ZULU,52,4,52.968411,1976,5540,0.356679
ZUNI,41,4,41.049617,1643,5540,0.296570
ZWEI,13,4,13.870041,346,5540,0.062455


In [143]:
clues_clean = clues_clean.merge(clues_ranks.reset_index()[["answer","clue_pct"]], on="answer")

In [144]:
clues_clean.sample(10)

,answer,clue_1,clue_2,clue_3,clue_4,clue_5,clue_6,clue_7,clue_8,clue_9,clue_10,clue_pct
8534,EDITOUT,Bleep,Bleep,Bleep,Bleep,Delete,Delete from a manuscript,"Delete, as from an article or video",Leave on the cutting room floor,Leave on the cutting room floor,"Remove, as an unnecessary line",0.750381
7187,DENALI,"*""Let's shake on it!""",Alaska vacation destination,Alaskan national park,Alternate name for Mount McKinley,Grizzly's milieu,Highest peak in Alaska,"Landmark sometimes called ""The Great One""",National park in Alaska,Park with a large caribou herd,Peak once known as Mount McKinley,0.923786
4127,BOAS,Big snakes,Burlesque scarves,Constricting snakes,Drag queens' neckwear,Feathery accessories,Feathery scarves,Feathery scarves,Feathery scarves,Showy neckwear,Squeezing snakes,0.693863
16776,MAILIN,Like many rebates,Like some ballots,Like some rebates,Like some rebates,Like some rebates,Like some rebates,Like some rebates,"Rebate card, e.g.",Submit via USPS,Submit with a stamp,0.110433
27103,SPEE,"German admiral defeated by the British, 1914",German vice admiral killed in W.W. I's Battle ...,Graf ___,Graf ___,Graf ___ (German battleship),Graf _____,"Graf _____ (German ""pocket battleship"")",Graf___,WWII battleship Graf ___,World War I German count,0.480505
6512,CRUNCHTIME,*When to pull an all-nighter,Critical juncture,Do-or-die moment,"Do-or-die moment, or what you must do to this ...",Period before a big deadline,Pressure-packed period,When a chocoholic is under pressure?,When the pressure's on,When the pressure's on,When to pull an all-nighter,0.125628
585,AENEID,Augustan masterpiece,Epic Virgil poem,Epic poem starting with the flight from Troy,Epic poem with Queen Dido,"Latin epic that begins ""I sing of arms and the...",Poem with the story of the Trojan horse,Roman classic,Virgil epic,Virgil opus,"Work that starts ""I sing of arms and the man""",0.927136
21527,PILL,Capsule alternative,It may be hard to swallow,It's taken in for treatment,Medicinal morsel,Medicinal tablet,Medicine tablet,Sort who's hard to tolerate,Tablet,Vitamin unit,Wet blanket,0.610289
12276,HEADERS,Banners at the top of webpages,Major tumbles,Plunges,Results of bad trips?,Some soccer shots,Some soccer shots,They often follow corner kicks,Trips might cause them,Trips might cause them,Trips might cause them,0.035473
24383,RUNWAY,Fashion show locale,Fashion show locale,Fashion show locale,Model path,Model's path,Modeling spot,Place for model citizens?,Takeoff site,View from a control tower,Where models model,0.588299


In [145]:
clues_clean.to_csv("clues_clean.csv")